# Video Tokenizer Training (Spacetime Vector Quantized Variational Autoencoder)

In [1]:
import torch 

from torch.utils.data import DataLoader
from torchvision.datasets import UCF101

import lightning as L
from spacetime.models.tokenizers import STVQVae

import wandb

In [3]:
wandb.login()

wandb: WARNING Calling wandb.login() after wandb.init() has no effect.


True

## Lightning module

We will use pytorch lightning to reduce boiler plate (there's a lot in previous notebooks, despite the centralized modules in `/src`)

In [4]:
class STVQVaeModule(L.LightningModule):
    def __init__(
        self,
        num_heads,
        d_model,
        num_layers,
        d_linear,
        codebook_size,
        latent_dim,
        patch_size,
        frame_height,
        frame_width,
        num_frames,
        num_linear_layers=2,
        num_groups=8,
        dropout=0.1,
        beta=0.25
    ):
        super().__init__()
        self.model = STVQVae(
            num_heads=num_heads,
            d_model=d_model,
            num_layers=num_layers,
            d_linear=d_linear,
            codebook_size=codebook_size,
            latent_dim=latent_dim,
            patch_size=patch_size,
            frame_height=frame_height,
            frame_width=frame_width,
            num_frames=num_frames,
            num_linear_layers=num_linear_layers,
            num_groups=num_groups,
            dropout=dropout
        )
        self.beta = beta

    def forward(self, inputs):
        return self.model(inputs)
    
    def configure_optimizers(self):
        return torch.optim.AdamW(self.model.parameters(), lr=3e-4)

    def training_step(self, batch, batch_idx):
        x, _ = batch
        x_pred, z_e, z_quantized = self(x)
        recon_loss = torch.nn.functional.mse_loss(x_pred, x)
        codebook_loss = torch.nn.functional.mse_loss(z_quantized, z_e.detach())
        commit_loss = torch.nn.functional.mse_loss(z_e, z_quantized.detach())
        loss = recon_loss + codebook_loss + (self.beta * commit_loss)

        self._log_losses(loss, recon_loss, codebook_loss, commit_loss, is_training=True)
        return loss

    def validation_step(self, batch, batch_idx):
        x, _ = batch
        x_pred, z_e, z_quantized = self(x)
        recon_loss = torch.nn.functional.mse_loss(x_pred, x)
        codebook_loss = torch.nn.functional.mse_loss(z_quantized, z_e.detach())
        commit_loss = torch.nn.functional.mse_loss(z_e, z_quantized.detach())
        loss = recon_loss + codebook_loss + (self.beta * commit_loss)

        self._log_losses(loss, recon_loss, codebook_loss, commit_loss, is_training=False)
        return loss
    
    def _log_losses(self, loss, recon_loss, codebook_loss, commit_loss, is_training=True):
        prefix = "train" if is_training else "val"
        log_on_step = True if is_training else False
        log_on_epoch = True

        # Lightning logging
        self.log(f"{prefix}_loss", loss, on_step=log_on_step, on_epoch=log_on_epoch, prog_bar=True, logger=True)
        self.log(f"{prefix}_recon_loss", recon_loss, on_step=log_on_step, on_epoch=log_on_epoch, prog_bar=False, logger=True)
        self.log(f"{prefix}_codebook_loss", codebook_loss, on_step=log_on_step, on_epoch=log_on_epoch, prog_bar=False, logger=True)
        self.log(f"{prefix}_commit_loss", commit_loss, on_step=log_on_step, on_epoch=log_on_epoch, prog_bar=False, logger=True)

        # Weights & Biases logging
        if wandb.run is not None:
            wandb.log({
                f"{prefix}_loss": loss.item(),
                f"{prefix}_recon_loss": recon_loss.item(),
                f"{prefix}_codebook_loss": codebook_loss.item(),
                f"{prefix}_commit_loss": commit_loss.item(),
            }, step=self.global_step)

    

## Load UCF101 Action Recognition dataset 

We use the UCF101 dataset which contains 13,320 videos from 101 action categories. This dataset is commonly used for benchmarking video action recognition models, such as basketball shooting, biking, diving, golf swinging, horse riding, and playing musical instruments.


We created a subset of UCF101 with only 10 classes for faster experimentation. The selected classes are:
ApplyEyeMakeup, ApplyLipstick, Archery, BabyCrawling, BalanceBeam, BandMarching, BaseballPitch, Basketball, BasketballDunk, and BenchPress.

In [ ]:
train_dataset = UCF101(
    root='./data/UCF-101-downsized',
    annotation_path='./data/ucfTrainTestlist',
    frames_per_clip=8,
    step_between_clips=8,  # non overlapping clips
    train=True,
)

test_dataset = UCF101(
    root='./data/UCF-101-downsized',
    annotation_path='./data/ucfTrainTestlist',
    frames_per_clip=8,
    step_between_clips=8,
    train=False,
)

print(f"Downsized train dataset size: {len(train_dataset)} clips")
print(f"Downsized test dataset size: {len(test_dataset)} clips")

In [6]:
import torch.nn.functional as F


def collate_ucf101(batch):
    # batch: list of (video, label, index) where label is detection labels 
    # and index is the index of the class for recognition
    xs, ys = [], []
    for v, _, l in batch:
        # v: T, H, W, C  (uint8)
        v = v.permute(0, 3, 1, 2)            # -> T, C, H, W
        v = v.float() / 255.0
        v = F.interpolate(v, size=(224, 224), mode='bilinear', align_corners=False)  # resize frames
        v = v.permute(1, 0, 2, 3).contiguous()  # -> C, F, H, W
        xs.append(v.clone())                  # new storage
        ys.append(int(l))
    return torch.stack(xs, 0), torch.tensor(ys, dtype=torch.long)

train_dataloader = DataLoader(
    train_dataset,
    batch_size=4,
    shuffle=True,
    num_workers=8,
    collate_fn=collate_ucf101,
    pin_memory=True,
)

test_dataloader = DataLoader(test_dataset, batch_size=2, shuffle=False, collate_fn=collate_ucf101)

In [ ]:
import os
os.environ["WANDB_SILENT"] = "true"  # for privacy / not pushing personal stuff to git

params = {
    "num_heads": 4,
    "d_model": 512,
    "num_layers": 4,
    "d_linear": 512,
    "codebook_size": 512,
    "latent_dim": 256,
    "patch_size": 16,
    "frame_height": 224,
    "frame_width": 224,
    "num_frames": 8,
    "num_linear_layers": 2,
    "num_groups": 8,
    "dropout": 0.1,
    "max_epochs": 10,
    "precision": 16,
    "batch_size": 4,
}

wandb.init(
    project="spacetime",
    name="stvqvae_train_run",
    config=params,
)

In [ ]:

lightning_timesformer = STVQVaeModule(
    num_heads=params["num_heads"],
    d_model=params["d_model"],
    num_layers=params["num_layers"],
    d_linear=params["d_linear"],
    codebook_size=params["codebook_size"],
    latent_dim=params["latent_dim"],
    patch_size=params["patch_size"],
    frame_height=params["frame_height"],
    frame_width=params["frame_width"],
    num_frames=params["num_frames"],
    num_linear_layers=params["num_linear_layers"],
    num_groups=params["num_groups"],
    dropout=params["dropout"],
)

wandb.watch(lightning_timesformer, log="gradients", log_freq=100)

trainer = L.Trainer(max_epochs=2, precision=16)
trainer.fit(model=lightning_timesformer, train_dataloaders=train_dataloader, val_dataloaders=test_dataloader)

wandb.finish()